In [1]:
from fame.experiments import exp_A_1, exp_A_2, exp_A_2_no_overwrite, exp_B_no_overwrite
from keras.models import load_model

import numpy as np

# check robustness because of numerical approximation error between solvers
from fame.abstract_domain.utils import check_is_robust 

/Users/ducoffe_m/Library/Python/3.9/lib/python/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.Op.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()
/Users/ducoffe_m/Library/Python/3.9/lib/python/site-packages/onnxscript/converter.py:820: FutureWarning: 'onnxscript.values.OnnxFunction.param_schemas' is deprecated in version 0.1 and will be removed in the future. Please use '.op_signature' instead.
  param_schemas = callee.param_schemas()


In [2]:
import pickle
filename = "gtsrb.pickle"
with open(filename, 'rb') as handle:
    data = pickle.load(handle)

In [3]:
DATASET = "GTSRB"
MODEL = "cnn"
eps = 0.01

channel = 3
data_format = "channels_last"
n_class = 10

In [4]:
"""
download and process GTSRB data.
"""
x_test, y_test = data['x_test'], data['y_test']
x_valid, y_valid = data['x_valid'], data['y_valid']
x_test = x_test.astype('float32') / 255


x_test = np.reshape(x_test, (-1, 3072))

In [5]:
k_model = load_model("./models/xairobas_gtsrb-cnn.keras")

In [6]:
k_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (1, 32, 32, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tsrb-cnn_conv2d_BiasAdd__6      │ (1, 3, 32, 32)         │             0 │
│ (Permute)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tsrb-cnn_conv2d_BiasAdd         │ (1, 4, 15, 15)         │           112 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tsrb-cnn_conv2d_1_BiasAdd       │ (1, 4, 7, 7)           │           148 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tsrb-cnn_conv2d_1_BiasAdd__12   │ (1, 7, 7, 4)           │             0 │
│ (Permute)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ tsrb-cnn_flatten_Reshape        │ (1, 196)               │             0 │
│ (Reshape)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 20)                │         3,940 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (1, 20)                │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, 10)                │           210 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,410 (17.23 KB)

 Trainable params: 4,410 (17.23 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
indices = [i for i in range(100) if not i in [0,\
1,2,4, 5, 8, 11, 16, 20, 22, 24, 31, 38, 42, 43, 44, 46, 49, 50, 52, 53, 55, 56,57, 58, 60, 62, 64, 67, 72, 74, 77, 78, 79, 81, 83, 88, 91, 92, 93]]

In [8]:
def is_robust(j):
    return check_is_robust(model=k_model, 
                    input_sample=x_test[j], 
                    eps=eps, 
                    channel=channel, 
                    data_format=data_format, 
                    n_class=n_class)

In [9]:
indices = [i for i in indices if not is_robust(i)]

In [10]:
len(indices)

60

In [11]:
from fame.batch_free.free import free_iteratively_k_features

In [12]:
dataframe_repository = "./results"

In [13]:
EXP = "A_1"
filename = "{}_{}_{}".format(DATASET, MODEL, EXP)
dataframe_filename_A1 = filename

In [14]:
i=0
dico_a_1 = exp_A_1(
        model=k_model,
        x_test=x_test,
        y_test=y_test,
        indices=indices,
        eps=eps,
        dataframe_repository=dataframe_repository,
        dataframe_filename="{}_v{}".format(dataframe_filename_A1, i),
        channel=channel,
        data_format=data_format,
        n_class=n_class,
        verbose=1,
    )

ongoing index 3
Restricted license - for non-production use only - expires 2026-11-23
milp time 8.501198053359985
greedy time 1.38326096534729
ongoing index 6
milp time 1.193030834197998
greedy time 1.1963448524475098
ongoing index 7
milp time 7.445049047470093
greedy time 1.6728711128234863
ongoing index 9
milp time 18.859217166900635
greedy time 1.2601399421691895
ongoing index 10
milp time 19.012810945510864
greedy time 1.615799903869629
ongoing index 12
milp time 2.6835620403289795
greedy time 1.0899670124053955
ongoing index 13
milp time 15.303842067718506
greedy time 1.2446739673614502
ongoing index 14
milp time 1.2033751010894775
greedy time 1.0551512241363525
ongoing index 15
milp time 12.535828828811646
greedy time 1.4404001235961914
ongoing index 17
milp time 13.039309024810791
greedy time 1.3622357845306396
ongoing index 18
milp time 17.06603217124939
greedy time 1.7461128234863281
ongoing index 19
milp time 11.513272047042847
greedy time 2.006650924682617
ongoing index 21
m